# ML Honors Assignment - 6

**Name:** Vaishnav Mankar

**Rollno:** 41252

---

**Problem Statement:** Write a program to recognize a document is positive or negative based on polarity words using suitable classification method.

**Dataset:** Twitter US Airline Sentiment (Kaggle — `Tweets.csv`), positive and negative tweets only

**Requirements:** Python 3, KaggleHub, NumPy, Pandas, Scikit-Learn

**Code:-**

In [ ]:
!pip install -q kagglehub

In [ ]:
import re
import numpy as np
import pandas as pd
import kagglehub
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")
df = pd.read_csv(f"{path}/Tweets.csv")[["text", "airline_sentiment"]]
df = df[df["airline_sentiment"].isin(["positive", "negative"])].copy()
print("Documents:", len(df))
print(df["airline_sentiment"].value_counts())

In [ ]:
def clean(text):
    text = re.sub(r"http\S+|@\w+", " ", text.lower())
    return re.sub(r"[^a-z\s]", " ", text)


df["clean"] = df["text"].apply(clean)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["airline_sentiment"], test_size=0.2, stratify=df["airline_sentiment"], random_state=42)

vectorizer = CountVectorizer(stop_words="english", min_df=2, binary=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB().fit(X_train_vec, y_train)
pred = model.predict(X_test_vec)
print(f"Test Accuracy: {accuracy_score(y_test, pred) * 100:.2f}%\n")
print(classification_report(y_test, pred))

In [ ]:
vocab = vectorizer.get_feature_names_out()
classes = list(model.classes_)
polarity = model.feature_log_prob_[classes.index("positive")] - model.feature_log_prob_[classes.index("negative")]
order = np.argsort(polarity)
print("Most negative words:", ", ".join(vocab[order[:10]]))
print("Most positive words:", ", ".join(vocab[order[::-1][:10]]))

In [ ]:
documents = [
    "The crew was friendly and the flight was perfect, thank you",
    "Terrible service, delayed for hours and my bag was lost",
]
for doc in documents:
    print(f"{model.predict(vectorizer.transform([clean(doc)]))[0]:>8}  <-  {doc}")